In [0]:
import requests,json
from pyspark.sql import SparkSession

#defining the Spark Session
spark = SparkSession.builder.appName("JobETL_Bronze").getOrCreate()

### BRONZE LAYER ---> Ingesting Data to Databricks

In [0]:

#defining the API Key
API_KEY = "<YOUR_SERPAPI_API_KEY>"
roles = ["Data Engineer","AI Data Engineer", "Cloud Data Engineer"]
location = "India"
all_jobs = []

for role in roles:
    params={
        "engine":"google_jobs",
        "q":role,
        "location":location,
        "api_key":API_KEY
    }
    res = requests.get("https://serpapi.com/search.json", params=params)
    jobs = res.json().get("jobs_results", [])
    # print(jobs)

    for job in jobs:
        job["search_role"] = role
    all_jobs.extend(jobs)

# Bronze layer data written from source to bronze table 

bronze_df = spark.createDataFrame(all_jobs)
bronze_df.write.format("delta").option("mergeSchema", "true").mode("append").saveAsTable("default.jobs_raw")
print("Data Written to Bronze Layer")
display(bronze_df)

### SILVER LAYER ---> Cleaning and Structuring

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import trim, upper, col

spark= SparkSession.builder.appName("JobETL_Silver").getOrCreate()

#reading Bronze Layer
print("Reading from Bronze Layer -------> Silver Layer")
df = spark.read.format("delta").table("default.jobs_raw")

#cleaning and Structuring
silver_df=df.selectExpr(
    "title",
    "company_name",
    "location",
    "description",
    "job_id",  
    "detected_extensions.posted_at as posted_at",
    "search_role",
    "source_link"
).dropna(subset=["title", "company_name", "location"])

#display(silver_df)

silver_df=silver_df.withColumn("company_name",trim(upper(col("company_name"))))
silver_df=silver_df.dropna(subset=["posted_at"])
silver_df.write.format("delta").option("mergeSchema", "true").mode("append").saveAsTable("default.jobs_silver")
print("Data Written to Silver Layer")
display(silver_df)


### Golden LAYER ---> Generate KPI Tables

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark=SparkSession.builder.appName("JobETL_Gold").getOrCreate()

df=spark.read.format("delta").table("default.jobs_silver")

#writing the golden layer to a table  

df.write.format("delta").option("mergeSchema", "true").mode("append").saveAsTable("default.jobs_golden")

#KPI 1 - Top Companies 

top_companies = df.groupBy("company_name").count().orderBy(col("count").desc())
display(top_companies)

#KPI - Jobs by City 
df.groupBy('location').agg(count('*').alias('job_count')).orderBy(col('job_count').desc())
display(top_companies)


In [0]:
from pyspark.sql.functions import *
from pyspark.sql import SparkSession

spark=SparkSession.builder.appName("JobETL_Gold").getOrCreate()

df=spark.read.format("delta").table("default.jobs_golden")
df.write.mode("append").saveAsTable("JobAggregatorTable")
print("✅Data Successfully Written to Table ---> JobAggregatorTable")